# Aula 56 - (20/10/2025)

In [ ]:
import tensorflow as tf
# Mostra os dispositivos para executar o modelo
print("Dispositivos de processamento: ", tf.config.list_physical_devices())

Dispositivos de processamento:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# True/false se tem GPU
print("Tem GPU? ", tf.test.is_gpu_available())

Tem GPU?  True


O que já fizemos

In [ ]:
# Fazendo imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import models, layers
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Tratar os dados

# Ler o CSV
dados = pd.read_csv("churn.csv")

# Preparar o enconding dos valores
encoder = LabelEncoder()
dados['Gender'] = encoder.fit_transform(dados['Gender'])
dados['Subscription Type'] = encoder.fit_transform(dados['Subscription Type'])
dados['Contract Length'] = encoder.fit_transform(dados['Contract Length'])

# Separar o X e o Y
X = dados.drop("Churn", axis=1).values
y = dados["Churn"].values

# Normalizar os dados
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Criar uma rede neural
modelo = models.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Saída binária (Se Churn ou não-Churn)
])

# Compilando o modelo
modelo.compile(optimizer='adam',
               loss='binary_crossentropy',
               metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Treinado
with tf.device('/GPU:0'):
  modelo.fit(X_train, y_train,
           epochs=30,
           batch_size=16,
           validation_split=0.2)

Epoch 1/30
2575/2575 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.8703 - loss: 0.3084 - val_accuracy: 0.9305 - val_loss: 0.1671
Epoch 2/30
2575/2575 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9278 - loss: 0.1646 - val_accuracy: 0.9423 - val_loss: 0.1337
Epoch 3/30
2575/2575 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9438 - loss: 0.1311 - val_accuracy: 0.9506 - val_loss: 0.1183
Epoch 4/30
2575/2575 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9541 - loss: 0.1120 - val_accuracy: 0.9565 - val_loss: 0.1057
Epoch 5/30
2575/2575 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9607 - loss: 0.0992 - val_accuracy: 0.9600 - val_loss: 0.0977
Epoch 6/30
2575/2575 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9616 - loss: 0.0945 - val_accuracy: 0.9604 - val_loss: 0.0945
Epoch 7/30
2575/2575 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9642 - loss: 0.0894 - val_accuracy: 0.9668 - val_loss: 0.0842
Epoch 8/30
2575/2575 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9691 - loss: 0.0798 -

In [ ]:
# Avaliação no meu conjunto de teste
loss, acc = modelo.evaluate(X_test, y_test)
print(f"Acurácia no meus dados de teste: {acc:.2f}")

# Salvar o modelo em arquivo
modelo.save("churn.h5")

403/403 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9857 - loss: 0.0373


Acurácia no meus dados de teste: 0.98


In [ ]:
from tensorflow.keras.models import load_model

modelo2 = load_model("churn.h5")

In [ ]:
exemplo = dados.drop("Churn", axis=1).iloc[0].values
resultado_esperado = dados["Churn"].iloc[0]
entrada = np.array(exemplo).reshape(1,-1)

print("Entrada:", entrada)
print("===================")

print("Resultado esperado: ", resultado_esperado)
predicao = modelo2.predict(entrada)
print("Minha previsão: ", predicao)

Entrada: [[  1  22   0  25  14   4  27   0   1 598   9]]
Resultado esperado:  1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Minha previsão:  [[1.]]


In [ ]:
x = [[  1,  22,   0,  25,  14,   4,  27,   0,   1, 598,   9]]

predicao = modelo2.predict(entrada)
print("Minha previsão: ", predicao)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Minha previsão:  [[1.]]


Criar uma API Rest para responder as chamadas

In [ ]:
from flask import Flask, request, jsonify
from tensorflow.keras.models import load_model
import numpy as np
import threading

app = Flask(__name__)

model = load_model('churn.h5')

@app.route('/')
def home():
  return jsonify({'message': 'API funcionando!'}) # API para validar o funcionamento do código

@app.route('/predict', methods=['POST'])
def predict():
  try:
    # Pegando dados para fazer a previsão
    data = request.get_json(force=True)

    # Pegar dados de dentro do campo dados e transformar em vetor
    dados_entrada = np.array(data['dados'])

    if dados_entrada.ndim == 1:
            dados_entrada = np.expand_dims(dados_entrada, axis=0)

    # Efetivamente chamando o modelo
    # Executando a previsão
    previsao = model.predict(dados_entrada)
    previsao = previsao.tolist()

    return jsonify({'previsao': previsao}) # Resposta da API se tudo OK

  except Exception as e:
    return jsonify({'error': str(e)}) # Retornando o erro que aconteceu durante a execução

def run():
  app.run(port=5000)

In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [ ]:
threading.Thread(target=run).start()

In [ ]:
!lt --port 5000 &
!curl https://loca.lt/mytunnelpassword

your url is: https://empty-breads-look.loca.lt
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 507ms/step


INFO:werkzeug:127.0.0.1 - - [21/Oct/2025 01:18:13] "POST /predict HTTP/1.1" 200 -


35.204.230.44

curl -X POST https://empty-breads-look.loca.lt/predict -H "Content-Type: application/json" -d "{\"dados\": [1, 22, 0, 25, 14, 4, 27, 0, 1, 598, 9]}"